# Aplicación simple con LangChain — VERSIÓN DETALLADA

_Qué es LangChain, sus abstracciones, LCEL y un mini-asistente de Q&A sobre nuestra propia documentación_

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering  
**Profesor:** Miguel Arquez

---

> 📘 **Versión detallada:** Este notebook expande el contenido sobre LangChain con análisis profundo de LCEL, arquitectura interna de RAG, estrategias de chunking, comparación de vector stores, y patrones avanzados de composición de cadenas.

![AI Engineering](assets/header.png)

## 1. ¿Qué es LangChain?

[LangChain](https://www.langchain.com/) es un **framework para construir aplicaciones con LLMs**. No es un modelo: es la "tubería" que conecta el modelo con prompts, fuentes de datos, herramientas, memoria y otros componentes.

Su propuesta de valor:
- **Abstracciones comunes** sobre múltiples proveedores (OpenAI, Anthropic, Gemini, Ollama, HuggingFace, …) → cambias el modelo cambiando una línea.
- **Composición declarativa** — encadenas componentes con el operador `|` (LCEL — LangChain Expression Language).
- **Ecosistema** — 600+ integraciones con vector stores, document loaders, herramientas, observabilidad.
- **Patrones listos** — RAG, agents, memory, structured output ya están resueltos.

## 2. Cuándo SÍ y cuándo NO usar LangChain

| Conviene | No conviene |
|---|---|
| Tu app encadena varios pasos (retriever → prompt → LLM → parser) | Una sola llamada `chat.completions.create` resuelve el problema |
| Quieres poder cambiar de proveedor sin reescribir | Estás 100% comprometido con un solo SDK |
| Necesitas integrar muchas fuentes (PDF, SQL, vector store, APIs) | Tu fuente es trivial (un string en memoria) |
| Estás haciendo RAG, agents, memory, tool use serios | Estás haciendo un wrapper fino sobre un endpoint |
| Quieres tracing/eval listos (LangSmith) | No te pesa montar tu propio logging |

> 💡 Regla práctica: **arranca sin LangChain** (con el SDK directo del proveedor). Adóptalo cuando sientas que estás reinventando sus abstracciones.

## 3. Las abstracciones core

| Abstracción | Qué representa | Ejemplo |
|---|---|---|
| `ChatModel` | un LLM con interfaz de chat | `ChatOpenAI(model='gpt-4o-mini')` |
| `Embeddings` | un modelo de embeddings | `OpenAIEmbeddings(model='text-embedding-3-small')` |
| `PromptTemplate` / `ChatPromptTemplate` | plantilla con variables | `'Resume: {texto}'` |
| `OutputParser` | parsea la salida del LLM | `JsonOutputParser`, `PydanticOutputParser` |
| `Runnable` | cualquier paso encadenable con `|` | `prompt | llm | parser` |
| `Retriever` | recupera documentos relevantes | `vectorstore.as_retriever()` |
| `VectorStore` | índice de embeddings | `Chroma`, `FAISS`, `Pinecone` |
| `Document` | texto + metadata | `Document(page_content=..., metadata=...)` |
| `Tool` / `Agent` | función ejecutable + lazo de razonamiento | `@tool def get_weather(...)` |
| `Memory` (legacy) / `History` | mantener contexto entre turnos | `RunnableWithMessageHistory` |

### LCEL — LangChain Expression Language

La sintaxis declarativa de LangChain. Encadenas componentes con `|`:

```python
chain = prompt | llm | output_parser
chain.invoke({'tema': 'embeddings'})
```

Ventajas: streaming, batching, async y observabilidad **gratis** sin tocar tu código.

---

## 3.1 LCEL en profundidad — composición funcional de cadenas

**LCEL (LangChain Expression Language)** es el corazón de LangChain moderno. Es una sintaxis declarativa para componer componentes usando el operador `|` (pipe).

### ¿Por qué LCEL?

**Antes de LCEL (legacy):**
```python
# Código verboso y poco flexible
chain = LLMChain(llm=llm, prompt=prompt)
result = chain.run(input_text)
```

**Con LCEL:**
```python
# Composición funcional clara
chain = prompt | llm | parser
result = chain.invoke({'input': input_text})
```

### Ventajas de LCEL

**1. Streaming gratis**

Cualquier cadena LCEL soporta streaming sin código adicional:

```python
for chunk in chain.stream({'input': 'Explica RAG'}):
    print(chunk, end='', flush=True)
```

**2. Batching automático**

Procesa múltiples inputs en paralelo:

```python
results = chain.batch([
    {'input': 'Tema 1'},
    {'input': 'Tema 2'},
    {'input': 'Tema 3'},
])
```

**3. Async nativo**

Versión asíncrona sin reescribir:

```python
result = await chain.ainvoke({'input': 'Tema'})
```

**4. Fallbacks**

Si un componente falla, prueba con otro:

```python
chain = prompt | llm.with_fallbacks([backup_llm]) | parser
```

**5. Observabilidad**

Con `LANGCHAIN_TRACING_V2=true`, cada paso se loggea automáticamente en LangSmith.

### Anatomía de un Runnable

Todo en LCEL es un `Runnable` — una interfaz con estos métodos:

| Método | Qué hace | Cuándo usar |
|---|---|---|
| `invoke(input)` | Ejecuta sincrónicamente | Default |
| `ainvoke(input)` | Ejecuta asincrónicamente | En apps async (FastAPI) |
| `stream(input)` | Retorna chunks a medida que se generan | UI interactiva |
| `astream(input)` | Stream asíncrono | Async + streaming |
| `batch(inputs)` | Procesa lista de inputs en paralelo | Batch processing |
| `abatch(inputs)` | Batch asíncrono | Async batch |

### Composición de Runnables

**Operador `|` (pipe):**

```python
chain = step1 | step2 | step3
```

Equivale a:

```python
def chain(input):
    x = step1.invoke(input)
    y = step2.invoke(x)
    z = step3.invoke(y)
    return z
```

**Diccionarios para inputs paralelos:**

```python
chain = {
    'contexto': retriever | format_docs,
    'pregunta': RunnablePassthrough()
} | prompt | llm
```

Ejecuta `retriever` y `RunnablePassthrough()` **en paralelo**, luego pasa ambos al prompt.

### RunnablePassthrough

Pasa el input sin modificarlo. Útil para preservar valores:

```python
{'pregunta': RunnablePassthrough()}
# input: "¿Qué es RAG?"
# output: {'pregunta': "¿Qué es RAG?"}
```

### RunnableLambda

Envuelve una función Python arbitraria:

```python
from langchain_core.runnables import RunnableLambda

def procesar(text):
    return text.upper()

chain = RunnableLambda(procesar) | llm
```

### RunnableParallel

Ejecuta múltiples runnables en paralelo:

```python
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    resumen=chain_resumen,
    traduccion=chain_traduccion,
    sentimiento=chain_sentimiento,
)

result = parallel.invoke({'text': 'Hola mundo'})
# {'resumen': '...', 'traduccion': '...', 'sentimiento': '...'}
```

### Ejemplo completo: cadena con branching

```python
from langchain_core.runnables import RunnableBranch

# Clasificar primero
clasificador = prompt_clasificar | llm | StrOutputParser()

# Diferentes cadenas según la clasificación
cadena_tecnica = prompt_tecnica | llm | parser_tecnico
cadena_casual  = prompt_casual  | llm | parser_casual

# Branch según el resultado del clasificador
chain = RunnableBranch(
    (lambda x: 'técnica' in x.lower(), cadena_tecnica),
    (lambda x: 'casual' in x.lower(), cadena_casual),
    cadena_casual  # default
)

full_chain = clasificador | chain
```

### Debugging de cadenas

**Ver el grafo de la cadena:**

```python
chain.get_graph().print_ascii()
```

**Output:**
```
           +-----------+
           | Prompt    |
           +-----------+
                 |
           +-----------+
           | LLM       |
           +-----------+
                 |
           +-----------+
           | Parser    |
           +-----------+
```

**Inspeccionar inputs/outputs intermedios:**

```python
# Agregar un callback para ver cada paso
from langchain_core.callbacks import StdOutCallbackHandler

chain.invoke({'input': 'test'}, config={'callbacks': [StdOutCallbackHandler()]})
```

In [ ]:
# Demostración de LCEL avanzado
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
import time

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)

print("="*70)
print("EJEMPLO 1: Composición básica con |")
print("="*70)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Eres un asistente conciso. Responde en máximo 2 frases.'),
    ('user', '{input}'),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({'input': '¿Qué es LCEL?'})
print(f"Resultado: {result}\n")

print("="*70)
print("EJEMPLO 2: Streaming token a token")
print("="*70)

print("Pregunta: Cuenta del 1 al 5 con una palabra por número.\n")
print("Respuesta: ", end='')
for chunk in chain.stream({'input': 'Cuenta del 1 al 5 con una palabra por número.'}):
    print(chunk, end='', flush=True)
print("\n")

print("="*70)
print("EJEMPLO 3: Batch processing (paralelo)")
print("="*70)

inputs = [
    {'input': '¿Qué es RAG?'},
    {'input': '¿Qué es un embedding?'},
    {'input': '¿Qué es LCEL?'},
]

start = time.time()
results = chain.batch(inputs)
elapsed = time.time() - start

for inp, res in zip(inputs, results):
    print(f"Q: {inp['input']}")
    print(f"A: {res[:80]}...\n")

print(f"Tiempo total: {elapsed:.2f}s (3 llamadas en paralelo)\n")

print("="*70)
print("EJEMPLO 4: RunnableParallel — ejecutar múltiples cadenas")
print("="*70)

# Definir 3 cadenas diferentes
prompt_resumen = ChatPromptTemplate.from_messages([
    ('system', 'Resume en una frase.'),
    ('user', '{text}'),
])

prompt_sentimiento = ChatPromptTemplate.from_messages([
    ('system', 'Clasifica el sentimiento como POSITIVO, NEGATIVO o NEUTRO.'),
    ('user', '{text}'),
])

prompt_idioma = ChatPromptTemplate.from_messages([
    ('system', 'Detecta el idioma del texto. Responde solo con el nombre del idioma.'),
    ('user', '{text}'),
])

# Ejecutar las 3 en paralelo
parallel_chain = RunnableParallel(
    resumen=prompt_resumen | llm | StrOutputParser(),
    sentimiento=prompt_sentimiento | llm | StrOutputParser(),
    idioma=prompt_idioma | llm | StrOutputParser(),
)

texto = "Me encanta LangChain! Es muy poderoso y fácil de usar."

start = time.time()
result = parallel_chain.invoke({'text': texto})
elapsed = time.time() - start

print(f"Texto: {texto}\n")
print(f"Resumen:     {result['resumen']}")
print(f"Sentimiento: {result['sentimiento']}")
print(f"Idioma:      {result['idioma']}")
print(f"\nTiempo: {elapsed:.2f}s (3 llamadas en paralelo)\n")

print("="*70)
print("EJEMPLO 5: RunnableLambda — funciones Python en la cadena")
print("="*70)

def contar_palabras(text):
    """Función Python arbitraria."""
    return {'text': text, 'n_palabras': len(text.split())}

def formatear_resultado(data):
    """Otra función Python."""
    return f"El texto tiene {data['n_palabras']} palabras: '{data['text']}'"

chain_con_lambda = (
    RunnableLambda(contar_palabras)
    | RunnableLambda(formatear_resultado)
)

result = chain_con_lambda.invoke("LangChain es un framework poderoso")
print(result)
print()

print("="*70)
print("EJEMPLO 6: Fallbacks — si falla, prueba con otro")
print("="*70)

# Simulamos un LLM que falla
class FakeLLMQueFalla:
    def invoke(self, *args, **kwargs):
        raise Exception("API caída!")
    
llm_principal = FakeLLMQueFalla()
llm_backup = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Cadena con fallback
from langchain_core.runnables import RunnableWithFallbacks

chain_con_fallback = prompt | RunnableWithFallbacks(
    runnable=llm_principal,
    fallbacks=[llm_backup]
) | StrOutputParser()

try:
    result = chain_con_fallback.invoke({'input': '¿Qué es un fallback?'})
    print(f"✅ Respuesta (usando backup): {result}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n💡 Observaciones:")
print("   • LCEL hace streaming, batching y async gratis")
print("   • RunnableParallel ejecuta múltiples cadenas en paralelo")
print("   • RunnableLambda envuelve funciones Python arbitrarias")
print("   • Fallbacks permiten resiliencia ante fallos")
print("   • Todo es composable con el operador |")


## 4. Setup

```bash
uv add langchain langchain-openai langchain-community langchain-text-splitters
# Para vector store local:
uv add langchain-chroma
```

Necesitas tu `OPENAI_API_KEY` en el entorno (igual que en notebooks anteriores).

In [ ]:
# Cargar API key desde .env si existe
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

if not os.environ.get('OPENAI_API_KEY'):
    print('⚠️  No hay OPENAI_API_KEY. Las celdas de ejemplo no se ejecutarán.')
else:
    print('✅ API key encontrada.')


## 5. Cadena más simple posible — Prompt + LLM + Parser

Vamos a construir el "Hello World" de LangChain: una cadena que toma un tema y devuelve una explicación corta. Luego añadiremos piezas.

In [ ]:
# uv add langchain langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Eres un profesor que explica en máximo 2 frases para principiantes.'),
    ('user',   'Explica qué es {tema}.'),
])

parser = StrOutputParser()

# La magia de LCEL: encadenar con `|`
chain = prompt | llm | parser

print(chain.invoke({'tema': 'el algoritmo K-Means'}))
print('---')
print(chain.invoke({'tema': 'la arquitectura Transformer'}))


**Lo que acabamos de obtener gratis:**
- `chain.batch([...])` — procesar varias entradas en paralelo.
- `chain.stream(...)` — recibir tokens a medida que se generan.
- `await chain.ainvoke(...)` — versión asíncrona.
- Observabilidad si configuras LangSmith (`LANGCHAIN_TRACING_V2=true`).

Sin LangChain tendrías que implementar batching, streaming y tracing tú mismo.

## 6. Salida estructurada con Pydantic

Patrón muy útil: forzar al LLM a devolver un objeto con campos concretos. Usamos un `PydanticOutputParser` o el método nativo `with_structured_output`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ResumenPaper(BaseModel):
    titulo:        str   = Field(description='Título del paper')
    area:          Literal['NLP', 'Visión', 'RL', 'Tabular', 'Otro']
    contribucion:  str   = Field(description='Contribución principal en una frase')
    puntuacion:    int   = Field(ge=1, le=5, description='Cuán importante es del 1 al 5')

# El método with_structured_output devuelve directamente el objeto Pydantic
llm_estructurado = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(ResumenPaper)

abstract = '''
We propose a new architecture, the Transformer, based solely on attention mechanisms,
dispensing with recurrence and convolutions entirely. Experiments on machine translation
show superior quality while being more parallelizable and requiring less time to train.
'''

resultado = llm_estructurado.invoke(f'Resume este abstract: {abstract}')
print(type(resultado).__name__)
print(f'Título      : {resultado.titulo}')
print(f'Área        : {resultado.area}')
print(f'Contribución: {resultado.contribucion}')
print(f'Puntuación  : {resultado.puntuacion}/5')


## 7. Aplicación — mini RAG sobre nuestra propia documentación

Un caso típico: queremos un asistente que responda preguntas usando el contenido de **nuestros propios documentos** (sin alucinar). El patrón es **RAG (Retrieval-Augmented Generation)**:

```
Pregunta → Embedding → Buscar K docs más similares → Pasarlos como contexto al LLM → Respuesta
```

Vamos a indexar el README del propio repo y preguntarle cosas.

In [ ]:
# uv add langchain-chroma langchain-community langchain-text-splitters
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 1. Cargar el README del repo
docs = TextLoader('../README.md').load()

# 2. Trocearlo en chunks pequeños (los LLMs trabajan mejor con contexto acotado)
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(docs)
print(f'Documentos: {len(docs)} | Chunks: {len(chunks)}')

# 3. Embeddings + vector store local (Chroma en memoria)
embeddings   = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore  = Chroma.from_documents(chunks, embeddings)
retriever    = vectorstore.as_retriever(search_kwargs={'k': 4})

# 4. Probemos solo el retriever
hits = retriever.invoke('¿Qué dataset usa el módulo de clustering?')
for i, h in enumerate(hits, 1):
    print(f'--- Hit {i} ---')
    print(h.page_content[:200], '...\n')


In [ ]:
# 5. Cadena completa: pregunta → retriever → prompt con contexto → LLM → respuesta
from langchain_core.runnables import RunnablePassthrough

prompt_rag = ChatPromptTemplate.from_messages([
    ('system',
     'Eres un asistente del curso DSRP Machine Learning Engineering. '
     'Responde la pregunta del usuario usando ÚNICAMENTE el contexto que se te da. '
     'Si la respuesta no está en el contexto, di "No lo sé". '
     'Sé conciso (máximo 4 frases).'),
    ('user', 'Contexto:\n{contexto}\n\nPregunta: {pregunta}'),
])

def formatear_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

rag_chain = (
    {'contexto': retriever | formatear_docs, 'pregunta': RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

preguntas = [
    '¿Qué módulos tiene el curso?',
    '¿Qué algoritmo usa el notebook de clustering?',
    '¿Cuál es la receta de la pizza margarita?',  # no está en el README
]
for p in preguntas:
    print(f'❓ {p}')
    print(f'   → {rag_chain.invoke(p)}\n')


**Qué pasó técnicamente:**
1. El retriever buscó los 4 chunks del README más cercanos en embedding a la pregunta.
2. Esos chunks se inyectaron como `{contexto}` en el prompt.
3. El LLM respondió **basándose solo en lo recuperado**.
4. Para preguntas fuera del corpus (la pizza), el sistema dijo "No lo sé" porque así se lo instruimos.

Eso es un sistema RAG en menos de 30 líneas. La misma estructura se escala a miles de PDFs cambiando solo el `DocumentLoader` y el vector store.

## 8. Cambiar de proveedor con una línea

Una de las grandes promesas de LangChain: el resto del código no cambia.

```python
# OpenAI → Anthropic
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5')

# OpenAI → Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

# OpenAI → Ollama local
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2')
```

La cadena `prompt | llm | parser` sigue funcionando exactamente igual.

---

## 7.1 RAG en profundidad — arquitectura y optimizaciones

### Anatomía completa de un sistema RAG

```
┌─────────────────────────────────────────────────────────────┐
│                    FASE 1: INDEXACIÓN                        │
│  (se hace una vez, offline)                                  │
├─────────────────────────────────────────────────────────────┤
│  Documentos → Loader → Splitter → Embeddings → Vector DB    │
│                                                              │
│  1. DocumentLoader: Lee PDFs, web, SQL, APIs, etc.          │
│  2. TextSplitter: Trocea en chunks (ej. 500 tokens)         │
│  3. Embeddings: Convierte cada chunk a vector               │
│  4. VectorStore: Indexa vectores para búsqueda rápida       │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    FASE 2: QUERY (runtime)                   │
│  (se hace cada vez que el usuario pregunta)                  │
├─────────────────────────────────────────────────────────────┤
│  Pregunta → Embedding → Similarity Search → Top-K chunks    │
│           → Inyectar en prompt → LLM → Respuesta            │
│                                                              │
│  1. Embed la pregunta del usuario                           │
│  2. Buscar los K chunks más similares (cosine similarity)   │
│  3. Construir prompt: system + chunks + pregunta            │
│  4. LLM genera respuesta basada en los chunks               │
└─────────────────────────────────────────────────────────────┘
```

### Estrategias de chunking

El **chunking** (troceado) es crítico — chunks muy grandes → contexto irrelevante, chunks muy pequeños → información fragmentada.

**1. Fixed-size chunking**

Trozos de tamaño fijo con overlap:

```python
RecursiveCharacterTextSplitter(
    chunk_size=500,      # caracteres por chunk
    chunk_overlap=50,    # overlap entre chunks consecutivos
)
```

**Ventajas:** Simple, predecible  
**Desventajas:** Puede cortar en medio de una idea

**2. Semantic chunking**

Trocea por significado (detecta cambios de tema):

```python
from langchain_experimental.text_splitter import SemanticChunker

SemanticChunker(
    embeddings=OpenAIEmbeddings(),
    breakpoint_threshold_type='percentile',  # o 'standard_deviation'
)
```

**Ventajas:** Chunks más coherentes  
**Desventajas:** Más lento (requiere embeddings)

**3. Document-specific chunking**

Para código, markdown, HTML:

```python
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    Language,
)

# Markdown: trocea por headers (##, ###)
MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'H1'), ('##', 'H2')])

# Código: respeta sintaxis
RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=500,
)
```

### Comparación de chunk sizes

| Chunk size | Ventajas | Desventajas | Cuándo usar |
|---|---|---|---|
| **Pequeño (200-300)** | Precisión alta, menos ruido | Contexto fragmentado | Búsqueda de hechos específicos |
| **Mediano (500-800)** | Balance | — | Default razonable |
| **Grande (1000-2000)** | Contexto completo | Más ruido, más caro | Documentos narrativos |

### Estrategias de retrieval

**1. Similarity search (default)**

Busca los K chunks más cercanos por cosine similarity:

```python
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
```

**2. MMR (Maximal Marginal Relevance)**

Balancea relevancia y diversidad (evita chunks muy similares):

```python
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 20, 'lambda_mult': 0.5}
)
```

**3. Similarity score threshold**

Solo devuelve chunks con score > umbral:

```python
retriever = vectorstore.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.7, 'k': 4}
)
```

**4. Compression (reranking)**

Recupera muchos chunks, luego un modelo los reordena por relevancia:

```python
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)
```

### Comparación de vector stores

| Vector Store | Tipo | Ventajas | Desventajas | Cuándo usar |
|---|---|---|---|---|
| **FAISS** | Local | Muy rápido, gratis | Solo en memoria, no escala | Prototipado, <1M docs |
| **Chroma** | Local/cloud | Fácil setup, persistencia | Menos features que otros | Prototipado, apps pequeñas |
| **Pinecone** | Cloud | Managed, escala a billones | Costo, vendor lock-in | Producción seria |
| **Weaviate** | Self-host/cloud | Open-source, features avanzados | Setup más complejo | Producción, control total |
| **Qdrant** | Self-host/cloud | Muy rápido, Rust | Comunidad más pequeña | Producción, performance |
| **pgvector** | Postgres | Usa tu DB existente | Menos optimizado que otros | Si ya tienes Postgres |

### Métricas de evaluación de RAG

**1. Retrieval metrics:**
- **Recall@K:** ¿Cuántos de los chunks relevantes están en los top-K?
- **Precision@K:** ¿Cuántos de los top-K son realmente relevantes?
- **MRR (Mean Reciprocal Rank):** Posición promedio del primer chunk relevante

**2. Generation metrics:**
- **Faithfulness:** ¿La respuesta se basa solo en los chunks recuperados?
- **Answer relevance:** ¿La respuesta responde la pregunta?
- **Context relevance:** ¿Los chunks recuperados son relevantes?

**Herramientas:**
- **RAGAS:** Framework de evaluación de RAG
- **LangSmith:** Evaluar con LLM-as-judge

### Optimizaciones avanzadas

**1. Hybrid search (keyword + semantic)**

Combina BM25 (keyword) con embeddings:

```python
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(docs)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vectorstore.as_retriever()],
    weights=[0.5, 0.5]
)
```

**2. Parent-child chunking**

Recupera chunks pequeños, pero pasa al LLM el chunk padre (más contexto):

```python
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=small_splitter,
    parent_splitter=large_splitter,
)
```

**3. Query rewriting**

Reescribe la pregunta del usuario para mejorar retrieval:

```python
query_rewriter = ChatPromptTemplate.from_messages([
    ('system', 'Reescribe esta pregunta para que sea más clara y específica.'),
    ('user', '{query}'),
]) | llm | StrOutputParser()

rag_chain = (
    {'query': query_rewriter}
    | {'context': retriever, 'query': RunnablePassthrough()}
    | prompt
    | llm
)
```

In [ ]:
# Comparación de estrategias de chunking y retrieval
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import matplotlib.pyplot as plt
import numpy as np

print("="*70)
print("EXPERIMENTO 1: Comparación de chunk sizes")
print("="*70)

# Documento de ejemplo
texto_largo = """
Machine Learning es una rama de la inteligencia artificial que permite a las computadoras 
aprender de datos sin ser explícitamente programadas. Los algoritmos de ML identifican 
patrones en los datos y hacen predicciones basadas en esos patrones.

Existen tres tipos principales de aprendizaje automático: supervisado, no supervisado y 
por refuerzo. El aprendizaje supervisado usa datos etiquetados para entrenar modelos. 
El aprendizaje no supervisado encuentra patrones en datos sin etiquetar. El aprendizaje 
por refuerzo aprende mediante prueba y error.

Los modelos de ML se usan en muchas aplicaciones: reconocimiento de voz, visión por 
computadora, sistemas de recomendación, detección de fraude, y más. El campo está en 
constante evolución con nuevas técnicas como deep learning y transfer learning.
""" * 3  # Repetir para tener más texto

doc = Document(page_content=texto_largo)

# Probar diferentes chunk sizes
chunk_sizes = [100, 300, 500, 1000]
resultados = {}

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.1),  # 10% overlap
    )
    chunks = splitter.split_documents([doc])
    resultados[size] = {
        'n_chunks': len(chunks),
        'avg_length': np.mean([len(c.page_content) for c in chunks]),
        'chunks': chunks,
    }
    print(f"\nChunk size {size}:")
    print(f"  Número de chunks: {len(chunks)}")
    print(f"  Longitud promedio: {resultados[size]['avg_length']:.0f} caracteres")

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Número de chunks
sizes = list(resultados.keys())
n_chunks = [resultados[s]['n_chunks'] for s in sizes]

ax1.bar(range(len(sizes)), n_chunks, color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_xticks(range(len(sizes)))
ax1.set_xticklabels([f'{s}' for s in sizes])
ax1.set_xlabel('Chunk size (caracteres)')
ax1.set_ylabel('Número de chunks')
ax1.set_title('Número de chunks por tamaño')
ax1.grid(True, alpha=0.3, axis='y')

for i, val in enumerate(n_chunks):
    ax1.text(i, val + 0.2, str(val), ha='center', fontweight='bold')

# Subplot 2: Trade-off
ax2.plot(sizes, n_chunks, marker='o', linewidth=2, markersize=8, color='tomato')
ax2.set_xlabel('Chunk size (caracteres)')
ax2.set_ylabel('Número de chunks')
ax2.set_title('Trade-off: Chunk size vs Número de chunks\\n(más chunks = más precisión pero más costo)')
ax2.grid(True, alpha=0.3)
ax2.invert_yaxis()  # Invertir para mostrar que menos chunks es "mejor" para costo

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("EXPERIMENTO 2: Comparación de estrategias de retrieval")
print("="*70)

# Crear un vector store con chunks medianos
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents([doc])

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(chunks, embeddings)

# Pregunta de prueba
query = "¿Qué tipos de aprendizaje automático existen?"

# Estrategia 1: Similarity search
retriever_similarity = vectorstore.as_retriever(search_kwargs={'k': 3})
docs_similarity = retriever_similarity.invoke(query)

print(f"\nPregunta: {query}\n")
print("Estrategia 1: Similarity Search (top-3)")
for i, doc in enumerate(docs_similarity, 1):
    print(f"  [{i}] {doc.page_content[:100]}...")

# Estrategia 2: MMR (diversidad)
retriever_mmr = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.5}
)
docs_mmr = retriever_mmr.invoke(query)

print("\nEstrategia 2: MMR (balance relevancia + diversidad)")
for i, doc in enumerate(docs_mmr, 1):
    print(f"  [{i}] {doc.page_content[:100]}...")

# Comparar overlap entre resultados
similarity_texts = set(d.page_content for d in docs_similarity)
mmr_texts = set(d.page_content for d in docs_mmr)
overlap = len(similarity_texts & mmr_texts)

print(f"\nOverlap entre estrategias: {overlap}/3 chunks iguales")

print("\n" + "="*70)
print("EXPERIMENTO 3: Análisis de costos de RAG")
print("="*70)

# Simular costos
PRECIO_EMBEDDING = 0.02 / 1_000_000  # text-embedding-3-small
PRECIO_LLM_INPUT = 0.15 / 1_000_000  # gpt-4o-mini
PRECIO_LLM_OUTPUT = 0.60 / 1_000_000

# Escenario: 10,000 documentos, 100,000 queries/mes
n_docs = 10_000
n_queries_mes = 100_000

# Costos de indexación (una vez)
for chunk_size in [300, 500, 1000]:
    chars_per_doc = 1000  # promedio
    n_chunks_total = (n_docs * chars_per_doc) / chunk_size
    tokens_embedding = n_chunks_total * (chunk_size / 4)  # ~4 chars/token
    
    costo_indexacion = tokens_embedding * PRECIO_EMBEDDING
    
    print(f"\nChunk size {chunk_size}:")
    print(f"  Chunks totales: {n_chunks_total:,.0f}")
    print(f"  Costo indexación: ${costo_indexacion:.2f} (una vez)")

# Costos de query (recurrente)
k = 4  # chunks recuperados por query
tokens_per_chunk = 100  # promedio
tokens_query = 20
tokens_output = 100

costo_por_query = (
    (tokens_query * PRECIO_EMBEDDING) +  # embed la query
    ((k * tokens_per_chunk + tokens_query) * PRECIO_LLM_INPUT) +  # LLM input
    (tokens_output * PRECIO_LLM_OUTPUT)  # LLM output
)

costo_mensual = costo_por_query * n_queries_mes

print(f"\nCosto por query: ${costo_por_query:.6f}")
print(f"Costo mensual ({n_queries_mes:,} queries): ${costo_mensual:.2f}")

print("\n💡 Lecciones clave:")
print("   • Chunk size pequeño: más chunks, más precisión, más costo")
print("   • Chunk size grande: menos chunks, más contexto, menos costo")
print("   • MMR balancea relevancia y diversidad (evita redundancia)")
print("   • Costo de RAG: indexación (una vez) + queries (recurrente)")
print("   • Optimizar k (chunks recuperados) reduce costo significativamente")


## 9. Patrones avanzados (mapa rápido)

- **Agents** — el LLM decide qué tools llamar y en qué orden. Hoy se construyen con [LangGraph](https://langchain-ai.github.io/langgraph/) (parte del mismo ecosistema), que da control fino sobre el ciclo de razonamiento.
- **Memory / History** — `RunnableWithMessageHistory` para mantener contexto multi-turno persistido en Redis, SQLite, etc.
- **Streaming + UI** — `chain.stream(...)` se integra trivialmente con FastAPI/Streamlit/Chainlit.
- **Evaluación** — [LangSmith](https://smith.langchain.com/) graba cada llamada, permite hacer evals con LLM-as-judge y comparar versiones de prompts.
- **Async + batch** — `ainvoke`, `abatch` para correr cientos de llamadas en paralelo respetando rate limits.

## 10. Buenas prácticas

1. **Empieza sin LangChain** y muévete cuando sientas la fricción de orquestar varios pasos.
2. **Usa LCEL** (`prompt | llm | parser`), no las clases `LLMChain` / `ConversationChain` antiguas (deprecadas).
3. **Tipa la salida** con `with_structured_output(PydanticModel)` siempre que vayas a parsear.
4. **Versiona tus prompts** — un `PromptTemplate` mal cambiado puede romper producción silenciosamente.
5. **Activa LangSmith desde el día 1** si la app es seria — sin tracing es muy difícil debuggear cadenas largas.
6. **Cuidado con la versión** de LangChain: el ecosistema se mueve rápido, pinea versiones en `pyproject.toml`.

## 11. Referencias

- LangChain docs: https://python.langchain.com/docs/introduction/
- LCEL — LangChain Expression Language: https://python.langchain.com/docs/concepts/lcel/
- LangChain conceptual guide: https://python.langchain.com/docs/concepts/
- LangGraph (agents): https://langchain-ai.github.io/langgraph/
- LangSmith (observabilidad y evals): https://docs.smith.langchain.com/
- Chroma DB: https://docs.trychroma.com/
- Crítica equilibrada: https://www.octomind.dev/blog/why-we-no-longer-use-langchain-for-building-our-ai-agents (lectura recomendada para entender los trade-offs)